In [1]:
import numpy as np

# 输入数据
x = np.array([2, 4, 6, 8])
gamma = 2
beta = 1
eps = 0

# 1. 计算批次均值
mu = np.mean(x)
# 2. 计算批次方差
var = np.var(x)
# 3. BN归一化公式
y = gamma * (x - mu) / np.sqrt(var + eps) + beta

# 打印输出
print(f"批次均值 μ = {mu}")
print(f"批次方差 σ² = {var}")
print(f"归一化后结果 y1,y2,y3,y4：")
for idx, val in enumerate(y, 1):
    print(f"y{idx} = {val:.4f}")

# 精确根式形式打印
sqrt5 = np.sqrt(5)
print("\n精确表达式：")
print(f"y1 = 1 - 6*√5/5 = {1 - 6*sqrt5/5:.4f}")
print(f"y2 = 1 - 2*√5/5 = {1 - 2*sqrt5/5:.4f}")
print(f"y3 = 1 + 2*√5/5 = {1 + 2*sqrt5/5:.4f}")
print(f"y4 = 1 + 6*√5/5 = {1 + 6*sqrt5/5:.4f}")

批次均值 μ = 5.0
批次方差 σ² = 5.0
归一化后结果 y1,y2,y3,y4：
y1 = -1.6833
y2 = 0.1056
y3 = 1.8944
y4 = 3.6833

精确表达式：
y1 = 1 - 6*√5/5 = -1.6833
y2 = 1 - 2*√5/5 = 0.1056
y3 = 1 + 2*√5/5 = 1.8944
y4 = 1 + 6*√5/5 = 3.6833


In [ ]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False):
        super().__init__()
        # 主分支：两层3×3卷积 + BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 1×1卷积维度适配shortcut
        self.conv_shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1) if use_1x1conv else None

    def forward(self, x):
        # 主分支前向
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        # 短路分支维度对齐
        shortcut_x = self.conv_shortcut(x) if self.conv_shortcut is not None else x

        # 残差相加 + 激活
        out = torch.relu(out + shortcut_x)
        return out


# ---------------------- 测试代码 ----------------------
if __name__ == "__main__":
    # 测试1：输入输出通道相同，不用1×1卷积
    res_block1 = Residual(in_channels=64, out_channels=64, use_1x1conv=False)
    x1 = torch.randn(2, 64, 32, 32)
    y1 = res_block1(x1)
    print(f"【无1×1卷积】输入shape:{x1.shape}, 输出shape:{y1.shape}")

    # 测试2：通道数变化，启用1×1卷积对齐维度
    res_block2 = Residual(in_channels=64, out_channels=128, use_1x1conv=True)
    x2 = torch.randn(2, 64, 32, 32)
    y2 = res_block2(x2)
    print(f"【启用1×1卷积】输入shape:{x2.shape}, 输出shape:{y2.shape}")